# [Experiment] Parameter-Matched KAN-TabNet | Step LR | Higgs Boson

### Overview
This notebook introduces the Kolmogorov-Arnold Network (KAN) architecture into the TabNet framework. The network is specifically configured to match the total trainable parameter count of the original vanilla baseline.

### Methodological Context: Strict Isolation
To precisely isolate the effects of the KAN architecture, we maintain the exact same StepLR optimization environment used in our vanilla control—a schedule chosen to strictly adhere to the original TabNet paper's experimental design. By holding the optimization thermodynamics constant and parameter-matching the models, we guarantee that any performance delta is driven purely by the mathematical flexibility of the B-splines, rather than a brute-force increase in structural capacity or a shifted learning rate.

### The Hypothesis
We hypothesize that substituting standard linear transformations with learnable activation functions on the network edges will fundamentally alter how the model routes and represents features. This notebook serves to evaluate whether this architectural shift improves predictive accuracy or alters convergence behavior when operating under the exact same foundational constraints defined by the original TabNet authors.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import pandas as pd
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import gc

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz'
df = pd.read_csv(url, header=None)

# Column 0 is the class label (1 for signal, 0 for background). Columns 1-28 are features.
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values.astype(int)

# Free up the massive raw DataFrame from memory
del df
gc.collect()

# divide dataset into 80% train, 20% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

# divide temp into 10% validation and 10% test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)

print(f"\nFinal Train shape: {X_train.shape}")
print(f"Final Valid shape: {X_valid.shape}")
print(f"Final Test shape:  {X_test.shape}\n")


Final Train shape: (8800000, 28)
Final Valid shape: (1100000, 28)
Final Test shape:  (1100000, 28)



### Model Training

In [5]:
MAX_EPOCHS = 500

In [6]:
# Hyperparameters from original paper (TabNet-S model).
# Notes:
# - momentum is 1 - 0.6 (paper m_B) to align with PyTorch's BatchNorm API.
# - scheduler step_size of 37 epochs approximates the paper's 20k iterations
#   (based on a batch size of 16384 and ~8.8M training samples).
paper_config = {
    'n_steps': 5,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-6,
    'momentum': 0.4,
    'optimizer_fn': torch.optim.Adam,
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'scheduler_params': dict(step_size=37, gamma=0.9),
    'mask_type': 'sparsemax'
}

clf_kan = TabNetClassifier(
    n_d=9, # adjusted so parameter count is matched with vanilla TabNet
    n_a=9, # adjusted so parameter count is matched with vanilla TabNet
    **paper_config,
    clip_value=2.,
    optimizer_params=dict(lr=0.0025), # reduced as 0.02 lr is too aggressive
    use_kan=True,
    kan_grid_size=3,
    kan_spline_order=3,
    seed=seed,
    verbose=25
)

In [7]:
# Hyperparameters from original paper (TabNet-S model).
paper_fit_config = {
    'batch_size': 16384,
    'virtual_batch_size': 512,
}

clf_kan.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=75,
    num_workers=os.cpu_count(),
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 0.6243  | valid_accuracy: 0.68909 |  0:01:55s
epoch 25 | loss: 0.5357  | valid_accuracy: 0.72515 |  0:48:31s
epoch 50 | loss: 0.50486 | valid_accuracy: 0.74519 |  1:34:12s
epoch 75 | loss: 0.48552 | valid_accuracy: 0.75958 |  2:20:01s
epoch 100| loss: 0.48004 | valid_accuracy: 0.76322 |  3:05:52s
epoch 125| loss: 0.47774 | valid_accuracy: 0.76505 |  3:51:33s
epoch 150| loss: 0.47613 | valid_accuracy: 0.76406 |  4:37:27s
epoch 175| loss: 0.47531 | valid_accuracy: 0.76565 |  5:23:08s
epoch 200| loss: 0.47433 | valid_accuracy: 0.7664  |  6:08:40s
epoch 225| loss: 0.47379 | valid_accuracy: 0.76686 |  6:54:40s
epoch 250| loss: 0.47368 | valid_accuracy: 0.76659 |  7:40:28s
epoch 275| loss: 0.47315 | valid_accuracy: 0.76601 |  8:26:24s
epoch 300| loss: 0.47246 | valid_accuracy: 0.76756 |  9:12:09s
epoch 325| loss: 0.47232 | valid_accuracy: 0.76735 |  9:58:05s
epoch 350| loss: 0.47207 | valid_accuracy: 0.76758 |  10:44:02s
epoch 375| loss: 0.47169 | valid_accuracy: 0.76791 |  

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_kan.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 78,854


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_kan.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.76834


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/higgs_boson/tables'
MODELS_DIR = f'{BASE_DIR}/results/higgs_boson/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "KAN-TabNet",
    "dataset": "Higgs Boson",
    "parameters": param_count,
    "scheduler": "StepLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_kan.best_epoch,
    "training_history": clf_kan.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/02_kan_param_matched_step_lr_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_kan.save_model(f'{MODELS_DIR}/02_kan_param_matched_step_lr_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/higgs_boson/tables/02_kan_param_matched_step_lr_metric.json
Successfully saved model at ./kan-tabnet-experiments/results/higgs_boson/models/02_kan_param_matched_step_lr_78k.zip
